# E2 · Coefficients with the wrong names: the column order after the ColumnTransformer

**Type**: Find the bug · **Level**: ●●○ independent · **Time**: about 30 minutes · **core path ★**

## Objective

Discover that a variable importance analysis gives a wrong answer because the names of the original columns were
matched to the coefficients in a different order from the one in which the `ColumnTransformer` produced them, and rebuild the right names.

## Dataset

`ds.loans(with_missing=False)`: four numeric columns (`age`, `income`, `job_tenure`, `amount`) and the target `defaulted`. A
`ColumnTransformer` standardizes `income` and `amount` first and then `age` and `job_tenure`: the order of the output columns is
**different** from the input one, but the number of columns is the same.

## Educational purpose

Understand that a transformer in scikit-learn returns an array **without names**, in an order that depends on how it is defined. And
learn `get_feature_names_out()` to recover them.

## Task

1. Run the colleague's cell and look at the variable it declares to be the most influential (`bug_variable`).
2. Rebuild the correct names with `ct.get_feature_names_out()` (removing the step prefix: `n.split("__")[-1]`) and save them in `correct_names`.
3. Build `coef_ok` (a `Series` with `correct_names` as index) and save `right_variable`, the variable with the largest coefficient **in absolute value**.
4. Save `income_sign` (`"positive"` or `"negative"`) and `amount_sign` according to the correct coefficients, and comment on them: do the signs make sense?

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

df = ds.loans(with_missing=False)
X = df[["age", "income", "job_tenure", "amount"]]
y = df["defaulted"]

ct = ColumnTransformer([("a", StandardScaler(), ["income", "amount"]),
                        ("b", StandardScaler(), ["age", "job_tenure"])])
Z = ct.fit_transform(X)
model = LogisticRegression(max_iter=2000).fit(Z, y)

## Your work

### The colleague's code

In [ ]:
coef_bug = pd.Series(model.coef_[0], index=X.columns)          # X.columns = age, income, job_tenure, amount
bug_variable = coef_bug.abs().idxmax()
print(coef_bug.round(2).to_string())
print("most influential variable according to the colleague:", bug_variable)

### Your fix

In [ ]:
correct_names = None
coef_ok = coef_bug
right_variable = bug_variable
income_sign = None
amount_sign = None

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='`correct_names = [n.split("__")[-1] for n in ct.get_feature_names_out()]`. The sign: `"positive" if coef_ok["income"] > 0 else "negative"`.'):
    check(correct_names == ["income", "amount", "age", "job_tenure"], "names rebuilt in the output order of the ColumnTransformer",
          "the order is: income, amount, age, job_tenure (get_feature_names_out, without prefix)")
    check(len(coef_ok) == 4 and list(coef_ok.index) == correct_names, "coef_ok has the correct names as index", "coef_ok = pd.Series(model.coef_[0], index=correct_names)")
    check(close(coef_ok.to_numpy(), model.coef_[0]), "the coefficients are the same: only the names change", "don't change the values, only the matching to the names")
    check(right_variable == coef_ok.abs().idxmax() and right_variable != bug_variable,
          "the most influential variable is a different one from the one declared by the colleague", "right_variable = coef_ok.abs().idxmax()")
    check(income_sign == "negative" and amount_sign == "positive",
          "a higher income lowers the risk, a higher amount raises it: sensible signs",
          "look at the sign of the coefficients of income and amount in coef_ok")
done()

---
The commented solution is in `solutions/E_find_the_bug/E2_wrong_coefficient_names.ipynb`. Open it after you pass the check, or after you have really tried.